# CNN for Classifying EMG Muscle Contractions 

In [ ]:
import os.path as op
import mne 
import os
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score, f1_score, classification_report  # Import necessary metrics
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from matplotlib import pyplot as plt
import tkinter as tk
import glob
import matplotlib

from scipy.fft import fft, ifft,fftfreq
from scipy.signal import welch, find_peaks 

from tensorflow.keras.models import Sequential  # Import Sequential model from TensorFlow Keras
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Input, Dropout  # Import necessary layers from TensorFlow Keras

matplotlib.use('QtAgg') 
mne.set_log_level("CRITICAL")

In [ ]:
# sanity check for columns to remove 
trigger_info = pd.read_csv("trigger_counts.csv")
print(trigger_info.loc[trigger_info["Trigger_number"] != 60, ["Subject", "Nap"]])

# Defining initial variables 

In [ ]:
to_keep=['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2','EOG1','EOG2','Corr', 'Zygo', 'Menton','Trigger']
eeg_ch= ['Fp1', 'Fp2', 'C3', 'Cz', 'C4', 'P3', 'Pz', 'P4', 'O1', 'O2']
emg_ch= ['Corr', 'Zygo', 'Menton'] # Menton is chin EMG for sleep scoring
eog_ch= ['EOG1', 'EOG2']
trigger_ch= ['Trigger']
subject_exclude = ["NL01SS", "NL02IF",
                   "NL05WW01", "RL12JL03", "RL07BR02", 
                   "RL11JH"] # trials to exclude based on trigger count 


inter_trigger_length = 30
num_epochs = 60 # 60 epochs for each session 

raw_path= "/Users/zeynepozkaya/Desktop/Consciousness_Research/Python_Scripts/EEG_data"
current_index=0
inter_trigger_length=10
window = 50 
step = 1 

In [ ]:
# pre-processes data for each subject and returns df with session divided into 60 epochs w meta-data attached 
def pre_process_subjets(subject,block):
    global raw_path
    global frq

    subject_name = subject + block 
    file= op.join(raw_path,'{}.edf'.format(subject_name))
    raw =  mne.io.read_raw_edf(file,preload=True)

    if 'Fp1/F3' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'Fp1/F3':'Fp1' ,'Fp2/F4':'Fp2'})
        
    if '36' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'36':'Corr' ,'37':'Zygo','38':'Menton'})

    if 'E1' in raw.info['ch_names']:
        mne.rename_channels(info=raw.info,mapping={'E1':'EOG1' ,'E2':'EOG2'})
    
    if 'Corru' in raw.info['ch_names']:
        raw.rename_channels({'Corru': 'Corr'})


    for ch in raw.info['ch_names']:
        if ch not in to_keep:
            raw.drop_channels([ch]) # only keeps channel that contains the trigger (where a stimulus was presented )

    raw.set_channel_types(mapping={'Corr':'emg','Zygo':'emg','Menton':'emg','Trigger':'stim','EOG1':'eog','EOG2':'eog'})

    raw= raw.resample(sfreq=250)

    filter_params_emg = {'lpass': 100,'hpass': 10,'notches': [50]}
    raw.filter(l_freq=filter_params_emg['hpass'],h_freq=filter_params_emg['lpass'],picks=emg_ch)
    filter_params_eeg_eog = {'lpass': 70,'hpass': 0.3,'notches': [50]}
    raw.filter(l_freq=filter_params_eeg_eog['hpass'],h_freq=filter_params_eeg_eog['lpass'],picks=eeg_ch+eog_ch)
    raw.notch_filter(filter_params_eeg_eog['notches'], method='fft', picks=emg_ch+eeg_ch+eog_ch)

    frq=raw.info['sfreq']

    # create dataframe based on events 
    events_all= mne.find_events(raw) # Get all the events (triggers) in your EEG data
    events= mne.pick_events(events_all,include=[201,202]) # Pick events of interest (where a stimulus was presented)

    df_triggers=pd.DataFrame(data=events,columns=['Time(Sample)','dunno','Trigger'])
    df_triggers.drop(columns=['dunno'],inplace=True)
    df_triggers['Time(s)']=df_triggers['Time(Sample)']/raw.info['sfreq']


    # incorporating meta_data 
    cols_inc = ["Subject","Nap_ID","Trigger", "Expected_Muscle","Nb_Corr","Nb_Zygo","Is_Correct"] # columns with relevant information from dataframe 
    trial_info = pd.read_csv("Trial_information_narcolepsy.csv", usecols=cols_inc) 
    expected_muscle = ["Corr", "Zygo"]


 
    epochs = mne.Epochs(raw, events, tmin=-0, tmax=9,baseline=None, detrend=0,
                    reject=None, preload=True, on_missing='warn')

   


    epochs.metadata = trial_info[(trial_info["Subject"] == subject) 
                                & (trial_info["Trigger"].isin([201.0, 202.0]))
                                & (trial_info["Nap_ID"] == int(block))]

    # adding metadata column for true activation
    # add another column for neither muscle being activated  
    true_activations = [] 
    for i in range(len(epochs.metadata)):
        true_ind = expected_muscle.index(epochs.metadata.iloc[i]["Expected_Muscle"]) 
        if (epochs.metadata.iloc[i]["Is_Correct"] == 0):
            if(epochs.metadata.iloc[i]["Nb_Corr"] < 3 and epochs.metadata.iloc[i]["Nb_Zygo"] < 3):
                true_activations.append("None")
            else:
                true_activations.append(expected_muscle[true_ind-1])
        else:
            true_activations.append(expected_muscle[true_ind])


    epochs.metadata["True_activation"] = true_activations
    return epochs, df_triggers 

In [ ]:
# gets features in a sliding window of size window samples with a step size of a certain number of samples 
def get_features(epoch):
     global frq
     global window 
     global step 

     # pad epoch to preserve sample number 
     pad_left  = window // 2
     pad_right = window - 1 - pad_left   
     
     epoch_padded = epoch
     epoch_padded = np.pad(epoch, (pad_left, pad_right), mode="edge")  

     # take sliding window 
     epoch_sw = np.lib.stride_tricks.sliding_window_view(epoch_padded,window)[::step]


     var = np.var(epoch_sw, axis=-1) # calculate variance over window 
     rms =  np.sqrt((1/window)*np.sum(epoch_sw**2, axis=-1)) # calculate rms over window  
     wl = np.sum(np.abs(np.diff(epoch_sw, axis=1)), axis=1) # calculate wl over window  
     mav = np.mean(np.abs(epoch_sw), axis=1)
     mavs = np.diff(mav)

     ''' 
     # frequency features 
    
     X = np.fft.rfft(epoch_sw, axis=1)
     PSD = (1/(frq*window)) * np.abs(X)**2
     cumulative = np.cumsum(PSD, axis=1)
     total_power = cumulative[:, -1]
     half_power = total_power / 2

     indices = (cumulative >= half_power[:, None]).argmax(axis=1)
     frequencies = np.fft.rfftfreq(window, 1/frq)

     fmd = frequencies[indices]
     '''

  
     return var, rms, wl, mavs


In [ ]:
def make_features_df(subject_epoch,subject,block):
    features = pd.DataFrame(
        index=range(num_epochs),
        columns=[
            "Subject",
            "Nap Number",
            "Triggers_Order_Nap", # epochs 
            "True_Muscle_Activated",
            "Num_Contractions_Zygo",
            "Num_Contractions_Corr",
            "WL_Zygo", # three features being used 
            "Var_Zygo",
            "RMS_Zygo", 
            "MAVS_Zygo",
            "WL_Corr",
            "Var_Corr",
            "RMS_Corr", 
            "MAVS_Corr",
            "Zygo", # processed EMG signal for epoch 
            "Corr"
        ]
        )   
    
    for t in range(len(subject_epoch)): 
        # extract epoch  
        epoch_zygo = np.squeeze(subject_epoch[t].get_data(picks=['Zygo']))
        epoch_corr = np.squeeze(subject_epoch[t].get_data(picks=['Corr']))

        epoch_var_zygo, epoch_rms_zygo, epoch_wl_zygo, epoch_mavs_zygo = get_features(epoch_zygo)
        epoch_var_corr, epoch_rms_corr, epoch_wl_corr, epoch_mavs_corr = get_features(epoch_corr)


        # fill dataframe 
        features.loc[t] = [
            subject, 
            int(block),
            t + 1,
            subject_epoch[t].metadata['True_activation'].iloc[0],
            subject_epoch.metadata.iloc[t]["Nb_Zygo"],
            subject_epoch.metadata.iloc[t]["Nb_Corr"],
            epoch_wl_zygo,
            epoch_var_zygo, 
            epoch_rms_zygo, 
            epoch_mavs_zygo,
            epoch_wl_corr,
            epoch_var_corr, 
            epoch_rms_corr, 
            epoch_mavs_corr,
            epoch_zygo, 
            epoch_corr
        ]
    
    return features 

Creating Dataframe

In [ ]:
# extracting features for classification 
i = 0 
excl = 0 # count which subjects had to be excluded 

features_results_mat = []

for root,dirs,files in os.walk(raw_path):
    for file in files:
        if "dpa" not in file and ".DS_Store" not in file: # so only take each subject once 
            subject = file.split(".")[0][:-2]
            block = file.split(".")[0][-2:]
            print(subject+block)


            # need to leave out these subjects 
            if (subject in subject_exclude or subject+block in subject_exclude):
                excl += 1 
                continue

            subject_epoch, _ = pre_process_subjets(subject,block)
            i += 1
            
            # make features data frame for each subject
            features = make_features_df(subject_epoch,subject,block)
            features_results_mat.append(features)

           


print(f"{i} trials processed, {excl} trials excluded")

In [ ]:
features_all = pd.concat(features_results_mat, ignore_index=True) # potentially save to excel and load in 
#features_all.to_excel("training_features_11032026.xlsx", index=False)
features_all.to_pickle("training_features_11032026.pkl")

In [ ]:
# load in features
#features_all = pd.read_excel("training_features_11032026.xlsx")
features_all = pd.read_pickle("training_features_11032026.pkl")

# GUI

In [ ]:
subject_plt = "RL08AE"
nap_plt = "03"
subject_df = features_all.loc[(features_all["Subject"] == subject_plt) & (features_all["Nap Number"] == int(nap_plt))] # extract subj info

In [ ]:
#GUI
current_index=0
inter_trigger_length=10
window = 50
step = 1

def plot_figure(t):
    global frq 
    global subject_df


    # Muscle Activated: {epochs[t].metadata['True_activation'].iloc[0]}, Corr:{epochs[t].metadata['Nb_Corr'].iloc[0]}, Zygo:{epochs[t].metadata['Nb_Zygo'].iloc[0]}")
    
    zygo_contractions = np.array(subject_df["Num_Contractions_Zygo"])[t]
    corr_contractions = np.array(subject_df["Num_Contractions_Corr"])[t]

    epoch_zygo = np.array(subject_df["Zygo"].tolist())
    epoch_zygo_rms = np.array(subject_df["Zygo"].tolist())
    epoch_zygo_var = np.array(subject_df["Var_Zygo"].tolist())
    epoch_zygo_wl = np.array(subject_df["WL_Zygo"].tolist())

    epoch_corr = np.array(subject_df["Corr"].tolist())
    epoch_corr_rms = np.array(subject_df["RMS_Corr"].tolist())
    epoch_corr_var = np.array(subject_df["Var_Corr"].tolist())
    epoch_corr_wl = np.array(subject_df["WL_Corr"].tolist())


    ymax_features = np.ceil(np.max([np.max(epoch_corr_wl[t]), np.max(epoch_zygo_wl[t])]) / 100) * 100
    ymax_emg = np.ceil(np.max([np.max(epoch_corr[t]), np.max(epoch_zygo[t])]) / 100) * 100

    fig, ax1 = plt.subplots(2, 1, figsize=(10, 5))

    # create twin axes
    ax2_corr = ax1[0].twinx()
    ax2_zygo = ax1[1].twinx()

    fig.suptitle(f"Epoch {t + 1}")

    # Corr subplot
    ax1[0].plot(epoch_corr[t], color="blue",alpha=0.5, label="Corr")
    ax2_corr.plot(epoch_corr_rms[t], label="rms", color="orange")
    ax2_corr.plot(epoch_corr_var[t], label="var", color="red")
    ax2_corr.plot(epoch_corr_wl[t], label="wl", color="green")

    ax1[0].set_ylim(-ymax_emg, ymax_emg)
    ax2_corr.set_ylim(0, ymax_features)

    print(zygo_contractions,corr_contractions)
    ax1[0].set_title(f"Zygo: {zygo_contractions}, Corr: {corr_contractions}")
    ax1[0].set_ylabel("Corr EMG [V]")
    ax2_corr.set_ylabel("Corr Features [V]")
    ax1[0].set_xlabel("Samples")

    # Zygo subplot
    ax1[1].plot(epoch_zygo[t], label="Zygo", alpha=0.5,color="black")
    ax2_zygo.plot(epoch_zygo_rms[t], label="rms", color="orange")
    ax2_zygo.plot(epoch_zygo_var[t], label="var", color="red")
    ax2_zygo.plot(epoch_zygo_wl[t], label="wl", color="green")

    ax1[1].set_ylim(-ymax_emg, ymax_emg)
    ax2_zygo.set_ylim(0, ymax_features)

    ax1[1].set_ylabel("Zygo EMG [V]")
    ax2_zygo.set_ylabel("Zygo Features [V]")
    ax1[1].set_xlabel("Samples")
    ax2_zygo.legend()  

 
 
    # Connect mouse click and key press events
    fig.canvas.mpl_connect('key_press_event', on_key)

    plt.tight_layout()
    plt.show()


# Keyboard press event handler
def on_key(event):
    global current_index, fig, features_all
    key = event.key
        
    if event.key == 'right':  # Move to next figure
        current_index = (current_index + 1) % len(subject_df)  # Loop to the start
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the next figure
    elif event.key == 'left':  # Move to previous figure
        current_index = (current_index - 1) % len(subject_df)  # Loop to the end
        plt.close()  # Close current figure
        plot_figure(current_index)  # Plot the previous figure

    elif event.key == 'escape':  
        print("Quitting the plot!")
        plt.close(fig)  
        

# Plot the first figure
plot_figure(current_index)

# CNN 

Defining Model 

In [36]:
def CNN_model(input_shape, num_classes,feature_num):
    model = Sequential([
        Input(shape=(2251, feature_num)) # NEED TO MAKE SHAPE A PARAMETER 
         # binary
    ])

    # Convolutional Layers - 
    model.add(Conv1D(32, kernel_size=3, activation='relu', input_shape=input_shape))  # Add a 1D convolutional layer with 32 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=2))  # Add a max pooling layer

    model.add(Conv1D(64, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer with 64 filters and ReLU activation
    model.add(MaxPooling1D(pool_size=1))  # Add another max pooling layer

    #model.add(Conv1D(128, kernel_size=3, activation='relu'))  # Add another 1D convolutional layer?

    # Flattening Layer
    model.add(Flatten())  # Flatten the output of the convolutional layers


    # Fully Connected Layers - connect each neuron of one layer to other neuron 
    model.add(Dense(128, activation='relu'))  # Add a fully connected layer with 128 neurons and ReLU activation
    model.add(Dropout(0.5)) # see if this changes anything? 
    model.add(Dense(num_classes, activation='softmax'))  # Add the output layer with softmax activation


    return model  # Return the compiled model

In [37]:
def plot_accuracy(model_history,i):
    
    plt.plot(model_history.history['accuracy'], label='accuracy')
    plt.plot(model_history.history['val_accuracy'], label = 'val_accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.ylim([0.5, 1])
    plt.title(f"Fold {i}")
    plt.legend(loc='lower right')

Training with KFolds 

In [54]:
# Define CNN model inputs 
subj_omit = False # omiting certain subjects on training - replace with string 

if (subj_omit):
    X_zygo = features_all.loc[features_all["Subject"] != subj_omit,[ "WL_Zygo", "Var_Zygo"]] 
    X_corr = features_all.loc[features_all["Subject"] != subj_omit,["WL_Corr", "Var_Corr"]]
else:
    X_zygo = features_all[[ "WL_Zygo", "Var_Zygo"]] 
    X_corr = features_all[["WL_Corr", "Var_Corr"]]


X_zygo = X_zygo.to_numpy()
X_zygo = np.array(X_zygo.tolist())
X_zygo = np.transpose(X_zygo, (0, 2, 1)) # setting dimensions 

X_corr = X_corr.to_numpy()
X_corr = np.array(X_corr.tolist())
X_corr = np.transpose(X_corr, (0, 2, 1)) # setting dimensions 

X = np.concatenate((X_zygo, X_corr), axis=0)
y = np.concatenate([features_all["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all["Num_Contractions_Corr"].astype(int).to_numpy()])       
   
y = np.concatenate([features_all["Num_Contractions_Zygo"].astype(int).to_numpy(),
                    features_all["Num_Contractions_Corr"].astype(int).to_numpy()])       
   
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # keep 20% purely for testing 

input_shape = X_train_full.shape[1:]  # Determine input shape based on the processed data with  
num_classes = len(np.unique(y))   # Determine the number of classes based on unique values in the target vector

input_shape = (input_shape[0], 1)  # Convert input shape to (input_shape[0], 1)

In [55]:
# Create CNN model using the adjusted input shape and number of classes
kf = KFold(n_splits=5, random_state = 42, shuffle=True) # 5 folds 
cvScores=[]

i = 1
for train_index, test_index in kf.split(X_train_full):
    print(f"Fold: {i} ==================================================================")
    
    X_train, X_val = X[train_index], X[test_index]
    y_train, y_val = y[train_index], y[test_index]

    model = CNN_model(input_shape, num_classes,2)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])  

    model_history_kfold = model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))
    plot_accuracy(model_history_kfold,i)
    
    scores = model.evaluate(X_test,y_test)
    cvScores.append(scores[1] * 100)


model_history = model.fit(X_train_full, y_train_full, epochs=10, validation_data=(X_test, y_test))

Fold: 1 ==================================================================


/Users/zeynepozkaya/anaconda3/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


ValueError: Expected `metrics` argument to be a list, tuple, or dict. Received instead: metrics=accuracy of type <class 'str'>

In [ ]:
# cross validation results 
avgScores = np.mean(cvScores)
stdScores = np.std(cvScores)

print(f"Average KFold Cross Validation Score: {avgScores}")
print(f"Standard Deviation KFold Cross Validation Score: {stdScores}")

In [ ]:
# full training results (test data not seen during cross val)
y_pred_train = model.predict(X_train)  
y_pred_train = np.argmax(y_pred_train, axis=1)   

# Predict on test data
y_pred_test = model.predict(X_test)   
y_pred_test = np.argmax(y_pred_test, axis=1)   

# Calculate accuracy
accuracy_training = accuracy_score(y_train, y_pred_train)   
accuracy_test = accuracy_score(y_test, y_pred_test)  

# Calculate F1 score
f1_training = f1_score(y_train, y_pred_train, average='weighted')  
f1_test = f1_score(y_test, y_pred_test, average='weighted')  

# Print accuracy and F1 score
print("Training Accuracy :", accuracy_training)  
print("Test Accuracy :", accuracy_test)  
print("Training F1 Score :", f1_training)   
print("Test F1 Score :", f1_test)   

In [ ]:
# plotting model accuracy 

plt.plot(model_history.history['accuracy'], label='accuracy')
plt.plot(model_history.history['val_accuracy'], label = 'val_accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.ylim([0.5, 1])
plt.legend(loc='lower right')